# NB3 — Huấn luyện DPO (notebook chính)

**Công nghệ:** TRL 1.13 `DPOTrainer`, LoRA mới trên mô hình SFT đã gộp, β=0.1, lr=5e-6.

> **Mục tiêu:** huấn luyện adapter DPO, vẽ **riêng** hai đường `rewards/chosen` và
> `rewards/rejected` trên cả tập huấn luyện và tập held-out, rồi tự chẩn đoán xem margin
> tăng theo kiểu nào (NB0 §5).

**Ba thay đổi so với lab cũ, đều ảnh hưởng tới kết quả:**
1. **Mô hình tham chiếu (reference) = mô hình SFT.** Lab cũ chồng LoRA DPO lên LoRA SFT rồi để TRL tắt
   adapter để lấy reference, tức là so với *mô hình gốc*. Ở đây mô hình đang học (policy) là
   `models/sft-merged` + LoRA mới (khởi tạo bằng 0), và
   `precompute_ref_log_probs=True` chấm mọi cặp trước bước cập nhật đầu tiên.
2. **lr = 5e-6.** 5e-7 là mức cho tinh chỉnh toàn bộ; với LoRA và ~100 bước, reward gần như đứng yên.
3. **Eval held-out.** Cặp eval không trùng câu hỏi với huấn luyện (NB2).

In [1]:
import sys
import time
import os
from contextlib import nullcontext
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lab22" / "config.py").exists())
sys.path.insert(0, str(ROOT))

import unsloth  # noqa: F401
import torch

from lab22 import config as C
from lab22 import data as D
from lab22 import modeling as MD

assert torch.cuda.is_available(), "DPO needs a CUDA GPU. See HARDWARE-GUIDE.md."
assert C.SFT_MERGED.exists(), f"Run NB1 first: {C.SFT_MERGED} missing"
assert (C.PREF_DIR / "train.parquet").exists(), "Run NB2 first"
C.ensure_dirs()
print(C.summary())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/day22-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


COMPUTE_TIER=T4 base=Qwen/Qwen3-0.6B max_len=768 seed=42
SFT saillab/alpaca-vietnamese-cleaned[:1000]  PREF sailor2/sea-ultrafeedback-onpolicy (Vietnamese) train=800 eval=100
DPO beta=0.1 lr=5e-06 epochs=1.0 loss=['sigmoid']


## 1. Mô hình đang học (policy) = SFT đã gộp + LoRA mới

Không có mô hình thứ hai trong VRAM. Log-prob của reference được tính một lần
trước khi huấn luyện (lúc LoRA còn bằng 0) rồi lưu lại, nên phần VRAM tăng thêm so
với SFT chủ yếu đến từ việc giữ cả câu chosen lẫn rejected trong một batch.

In [2]:
model, tokenizer = MD.load_model(C.SFT_MERGED)
model = MD.add_lora(model)

from datasets import Dataset

train_ds = Dataset.from_parquet(str(C.PREF_DIR / "train.parquet"))
eval_ds = Dataset.from_parquet(str(C.PREF_DIR / "eval.parquet"))
print(f"train={len(train_ds)} eval={len(eval_ds)}  columns={train_ds.column_names}")

==((====))==  Unsloth 2026.10.2: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA GeForce RTX 3050 Ti Laptop GPU. Num GPUs = 1. Max memory: 4.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu118. CUDA: 8.6. CUDA Toolkit: 11.8. Triton: 3.3.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/310 [00:22<1:53:28, 22.03s/it]

Loading weights:   1%|          | 3/310 [00:22<30:13,  5.91s/it]  

Loading weights:   1%|▏         | 4/310 [00:22<20:24,  4.00s/it]

...

Loading weights: 100%|██████████| 310/310 [01:17<00:00,  4.02it/s]

The tokenizer you are loading from '/lab/models/sft-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


The tokenizer you are loading from '/lab/models/sft-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Unsloth 2026.10.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


train=800 eval=100  columns=['prompt', 'chosen', 'rejected', 'chat_template_kwargs']


## 2. Huấn luyện

In [3]:
from trl import DPOTrainer

args = MD.dpo_config(C.ADAPTERS / "dpo-checkpoints")
print(f"loss_type={args.loss_type} beta={args.beta} lr={args.learning_rate} "
      f"precompute_ref={args.precompute_ref_log_probs} max_length={args.max_length}")

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)
torch.cuda.reset_peak_memory_stats()
training_started = time.perf_counter()
save_on_cpu = os.environ.get("DPO_SAVE_ON_CPU", "0") == "1"
with torch.autograd.graph.save_on_cpu(pin_memory=True) if save_on_cpu else nullcontext():
    result = trainer.train()
train_seconds = time.perf_counter() - training_started
final_eval = trainer.evaluate()
print(f"train loss {result.training_loss:.4f} · held-out reward accuracy "
      f"{final_eval.get('eval_rewards/accuracies', float('nan')):.3f}")

Unsloth: reducing dataset_num_proc 8 -> 2 to fit free memory (~1GB per worker). Set UNSLOTH_DATASET_NUM_PROC to override.
loss_type=['sigmoid'] beta=0.1 lr=5e-06 precompute_ref=True max_length=768


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 10,092,544 of 616,235,008 (1.64% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
25,0.697521,0.684431,1.841500,145434.000000,-2.620812,-2.598934,0.571699,0.049234,0.023134,0.530000,0.026100,-491.565000,-403.996782
50,0.680430,0.676025,1.842079,278676.000000,-2.623930,-2.601756,0.570993,0.104603,0.061297,0.520000,0.043306,-491.011318,-403.615158
75,0.670727,0.664908,1.841406,423394.000000,-2.624024,-2.601868,0.570811,0.152816,0.083557,0.640000,0.069260,-490.529181,-403.392558
100,0.677576,0.673746,1.842658,557774.000000,-2.623602,-2.602575,0.571085,0.143350,0.092542,0.610000,0.050808,-490.623845,-403.302704


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.677576,0.673746,100,1.842658,557774.000000,-2.623602,-2.602575,0.571085,0.143350,0.092542,0.610000,0.050808,-490.623845,-403.302704


train loss 0.6887 · held-out reward accuracy 0.610


## 3. Đường cong reward (sản phẩm nộp `03-dpo-reward-curves.png`)

Implicit reward = β·log(π/π_ref) nên bắt đầu ở 0. Đọc đường *chosen*, không chỉ margin:
- chosen ↑, rejected ↓ → đúng ý đồ;
- chosen ↓, rejected ↓ nhanh hơn → **likelihood displacement** (Razin et al. 2024).

Loss ở bước đầu phải gần 0.693 (NB0 §3). Nếu không, reference đang sai.

In [4]:
train_hist = MD.reward_history(trainer.state.log_history)
eval_hist = MD.reward_history(trainer.state.log_history, prefix="eval_")
title = f"DPO · {C.COMPUTE_TIER} · β={C.DPO_BETA} · lr={C.DPO_LR} · {'+'.join(C.DPO_LOSS)}"
MD.plot_rewards(train_hist, eval_hist, title, C.SCREENSHOTS / "03-dpo-reward-curves.png")

first_loss = next((r["loss"] for r in trainer.state.log_history if "loss" in r), None)
print(f"first logged loss: {first_loss}")

first logged loss: 0.6871114730834961


In [5]:
label, message = MD.diagnose(eval_hist if len(eval_hist) >= 2 else train_hist)
print(f"[{label}] {message}")

[INTENDED] Chosen +0.147 up, rejected +0.090, margin +0.057.


## 4. Lưu adapter + metrics

In [6]:
import json

trainer.model.save_pretrained(str(C.DPO_ADAPTER))
tokenizer.save_pretrained(str(C.DPO_ADAPTER))
D.save_split_fingerprint(C.PREF_DIR, C.DPO_ADAPTER)  # NB4 refuses a held-out set this adapter saw


def last(df, col):
    return float(df[col].iloc[-1]) if not df.empty and col in df else None


metrics = {
    "compute_tier": C.COMPUTE_TIER,
    "base_model": C.BASE_MODEL,
    "reference": "models/sft-merged (precomputed)",
    "pref_dataset": C.PREF_DATASET,
    "beta": C.DPO_BETA,
    "lr": C.DPO_LR,
    "loss_type": C.DPO_LOSS,
    "epochs": C.DPO_EPOCHS,
    "train_pairs": len(train_ds),
    "eval_pairs": len(eval_ds),
    "max_length": C.MAX_LEN,
    "seed": C.SEED,
    "train_seconds": train_seconds,
    "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024**3,
    "optimizer_steps": trainer.state.global_step,
    "saved_tensors_on_cpu": save_on_cpu,
    "log_history": trainer.state.log_history,
    "final_train_loss": float(result.training_loss),
    "first_logged_loss": first_loss,
    "end_chosen_reward": last(train_hist, "rewards/chosen"),
    "end_rejected_reward": last(train_hist, "rewards/rejected"),
    "end_reward_gap": last(train_hist, "rewards/margins"),
    "eval_chosen_reward": final_eval.get("eval_rewards/chosen"),
    "eval_rejected_reward": final_eval.get("eval_rewards/rejected"),
    "eval_reward_gap": final_eval.get("eval_rewards/margins"),
    "eval_reward_accuracy": final_eval.get("eval_rewards/accuracies"),
    "diagnosis": label,
}
(C.DPO_ADAPTER / "dpo_metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

Unsloth: Restored added_tokens_decoder metadata in /lab/adapters/dpo/tokenizer_config.json.


{
  "compute_tier": "T4",
  "base_model": "Qwen/Qwen3-0.6B",
  "reference": "models/sft-merged (precomputed)",
  "pref_dataset": "sailor2/sea-ultrafeedback-onpolicy",
  "beta": 0.1,
  "lr": 5e-06,
  "loss_type": [
    "sigmoid"
  ],
  "epochs": 1.0,
  "train_pairs": 800,
  "eval_pairs": 100,
  "max_length": 768,
  "seed": 42,
  "train_seconds": 1316.970919334,
  "peak_vram_gb": 3.316309928894043,
  "optimizer_steps": 100,
  "saved_tensors_on_cpu": true,
  "log_history": [
    {
      "loss": 0.6871114730834961,
      "grad_norm": 17.401988983154297,
      "learning_rate": 2.0000000000000003e-06,
      "entropy": 1.9408343613147736,
      "num_tokens": 26931.0,
      "logits/chosen": -2.6503197402894654,
      "logits/rejected": -2.5814556601542065,
      "mean_token_accuracy": 0.563360869139433,
      "rewards/chosen": 0.03012734481599182,
      "rewards/rejected": 0.005733479859190993,
      "rewards/accuracies": 0.475,
      "rewards/margins": 0.02439386521000415,
      "logps/chosen

## 5. Ghi chú vibe-coding: quét β (+6 độ chặt chẽ)

`make beta-sweep` chạy `scripts/train_dpo.py` với β ∈ {0.05, 0.1, 0.5} và lưu vào
`adapters/dpo-b*/`. `python scripts/eval_judge.py --plot-sweep` vẽ β theo margin held-out.

**Đoán trước khi xem:** β lớn giữ mô hình đang học (policy) gần reference hơn. Margin tính bằng β·log-ratio
sẽ thay đổi thế nào? Còn độ chính xác reward?

**Tiếp theo:** NB3b (so sánh RPO / SimPO-norm / LD-DPO / ORPO) hoặc NB4 (đánh giá).